# IE4422: Browser science tutor on Colab

Run cells 1–8 to set up and open the tutor. Questions, hints, and answer checks all happen in the UI. Cells 9–11 provide logs, shutdown, and result downloads. No local GPU or Hugging Face account is needed for these public downloads.

The UI uses the same evidence extractor as Experiment D: up to 10 brief numbered observations with types and locations, under a 320-token limit. The prompt excludes inference; evidence remains unverified.

Keep this notebook open while using the UI. See the bundled notebooks/README.md and the [Colab FAQ](https://research.google.com/colaboratory/faq.html) for runtime eligibility and storage details.


## 1. Check the cloud runtime

Select Runtime → Change runtime type → GPU, then run this cell. Use Python 3.12+ and a CUDA GPU; a TPU does not work with this adapter.


In [ ]:
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import subprocess
import sys
import zipfile

if not Path('/content').is_dir() or not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook in Google Colab, not a local notebook kernel.')
if sys.version_info < (3, 12):
    raise RuntimeError('Select a current Colab Python 3.12+ runtime for the shared dependency pins.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU, then reconnect and rerun.')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Local disk free (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))


## 2. Upload and extract the source ZIP

For the minimum-sufficient-observations prompt update, keep this notebook and upload the new IE4422_Source.zip. Leave UPLOAD_NEW_SOURCE=True. Replacing source stops the previous UI server; rerun cells 3-8 to launch it with the updated extractor. No notebook code changed for this update. A fresh runtime is recommended when replacing an older bundle.


In [ ]:
WORK = Path('/content/IE4422')
SOURCE_ZIP = Path('/content/IE4422_Source.zip')
UPLOAD_NEW_SOURCE = True  # @param {type:"boolean"}
if UPLOAD_NEW_SOURCE or not SOURCE_ZIP.is_file():
    if globals().get('UI_SERVER') is not None:
        UI_SERVER.stop()  # Release the old worker before replacing its source files.
        UI_SERVER = None  # Recreate the launcher from the newly uploaded source.
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one IE4422 source ZIP.')
    SOURCE_ZIP.write_bytes(uploaded[archives[0]])

# Validate paths before extracting the package. The ZIP contains source only.
with zipfile.ZipFile(SOURCE_ZIP) as source:
    names = set()
    for info in source.infolist():
        path = PurePosixPath(info.filename)
        if (not path.parts or path.parts[0] != 'IE4422' or path.is_absolute()
                or '..' in path.parts or '\\' in info.filename or ':' in info.filename
                or info.filename in names or stat.S_ISLNK(info.external_attr >> 16)
                or not (Path('/content') / info.filename).resolve().is_relative_to(WORK.resolve())):
            raise ValueError(f'Unexpected ZIP member: {info.filename}')
        names.add(info.filename)
    if 'IE4422/bundle_info.json' not in names:
        raise ValueError('Use the supplied IE4422_Source.zip, not a general repository download.')
    source.extractall('/content')
sys.path.insert(0, str(WORK))
import importlib
import notebooks.colab_support as storage
importlib.reload(storage)  # Pick up the helper from the newly uploaded bundle.
from notebooks.colab_support import (read_json, write_json, archive_prepared,
                                    restore_prepared, prepared_signature, prepared_files, run_visible)
bundle = read_json(WORK / 'bundle_info.json')
print('Source extracted:', WORK)
print('Bundle created:', bundle['created_at_utc'])


## 3. Choose persistence settings

With USE_DRIVE=True, prepared data and UI results persist in My Drive/IE4422. SAVE_MODEL_TO_DRIVE also retains the pinned checkpoint. REBUILD_DATA repeats full preparation. Without Drive, download results before the temporary runtime disappears.


In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}
SAVE_MODEL_TO_DRIVE = True  # @param {type:"boolean"}
REBUILD_DATA = False  # @param {type:"boolean"}
PERSISTENT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/IE4422')
    PERSISTENT.mkdir(parents=True, exist_ok=True)
print('Persistent folder:', PERSISTENT or 'disabled; download results at the end')


## 4. Set up one project environment

This creates a project environment that reuses Colab's CUDA wheels, installs compatible project dependencies, checks model adapter imports (including qwen_vl_utils), and checks CUDA before the first hint request. No model is loaded by this check. Setup avoids ensurepip and does not modify the notebook kernel. Output and errors are shown live.


In [ ]:
if globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()  # Do not change an environment used by a running worker.
ENVIRONMENT = Path('/content/ie4422_env')
run_visible([sys.executable, str(WORK / 'notebooks/colab_support.py'),
                'setup', str(WORK), str(ENVIRONMENT)], cwd=WORK)
PYTHON = ENVIRONMENT / 'bin/python'
print('Project Python:', PYTHON)


## 5. Check the packaged UI code

Run the packaged tests without loading a model. These cover the tutor branches, dataset boundaries, notebook cells, and server lifecycle.


In [ ]:
run_visible([str(PYTHON), '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=WORK)


## 6. Prepare and finalize ScienceQA

Restore matching prepared data from Drive or prepare the pinned ScienceQA release. Then validate the saved visual-need audit and rebuild the finalized selection. Public question inputs and reference material remain in separate exports. The UI offers train and validation only.


In [ ]:
if globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()  # Do not rebuild exports while the UI is serving them.
signature = prepared_signature(WORK)
local_marker = WORK / 'tmp/prepared_signature.json'
local_archive = WORK / 'tmp/scienceqa_prepared.zip'
exports = [WORK / name for name in prepared_files()]
ready = (not REBUILD_DATA and local_marker.is_file()
         and read_json(local_marker) == signature and all(path.is_file() for path in exports))
if not ready and not REBUILD_DATA and PERSISTENT and (PERSISTENT / local_archive.name).is_file():
    local_archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PERSISTENT / local_archive.name, local_archive)
    ready = restore_prepared(WORK, local_archive)
    print('Restored matching prepared and audited data from Drive.')
if not ready:
    run_visible([str(PYTHON), 'prepare_data.py'], cwd=WORK)
# Always validate the saved decisions and rebuild the audited exports before use.
run_visible([str(PYTHON), 'finalize_data.py'], cwd=WORK)
if not ready:
    archive_prepared(WORK, local_archive)
    if PERSISTENT:
        pending = PERSISTENT / (local_archive.name + '.partial')
        shutil.copy2(local_archive, pending)
        pending.replace(PERSISTENT / local_archive.name)
    print('Saved a reusable archive including the audited final set.')
write_json(local_marker, signature)
audit = read_json(WORK / 'reports/scienceqa/visual_audit.json')
print('Audited final counts:', audit['final_counts'])


## 7. Download/restore the fixed Qwen checkpoint

The pinned checkpoint is about 7.5 GB before quantization. A complete matching Drive copy can be reused. Downloading does not load the model into GPU memory; the UI loads it on the first hint or answer check.


In [ ]:
model_command = [str(PYTHON), str(WORK / 'notebooks/colab_support.py'), 'model', str(WORK)]
if PERSISTENT and SAVE_MODEL_TO_DRIVE:
    model_command.extend(['--persistent', str(PERSISTENT / 'models')])
run_visible(model_command, cwd=WORK)


## 8. Start the browser UI

This starts ui_server.py and embeds it through Colab's HTTPS port proxy. Choose a question, request a hint, or submit an answer and reasoning in the UI. Both hint requests and answer checks use the shared numbered-observation evidence pass. Rerun to reuse the same server/model; set RESTART_UI=True to replace it. Results are copied to Drive when enabled. Use a runtime eligible for web-UI interaction under the Colab FAQ; embedding does not change those terms.


In [ ]:
UI_PORT = 8765  # @param {type:"integer"}
RESTART_UI = False  # @param {type:"boolean"}
from google.colab import output
from urllib.parse import urlsplit
import importlib
import notebooks.colab_ui as colab_ui
importlib.reload(colab_ui)
if globals().get('UI_SERVER') is None:
    UI_SERVER = colab_ui.NotebookUIServer()
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({int(UI_PORT)}, {{cache: false}})')
parsed_proxy = urlsplit(proxy_url)
if parsed_proxy.scheme != 'https' or not parsed_proxy.netloc:
    raise RuntimeError('Colab did not return an HTTPS port proxy. Reconnect and rerun this cell.')
proxy_origin = f'{parsed_proxy.scheme}://{parsed_proxy.netloc}'
output_origin = output.eval_js('window.location.origin')
status = UI_SERVER.start(
    PYTHON, WORK, port=int(UI_PORT), proxy_origin=proxy_origin,
    frame_ancestors=('https://colab.research.google.com', output_origin),
    persistent_output=PERSISTENT / 'runs/ui' if PERSISTENT else None,
    restart=RESTART_UI,
)
print('Server ready. Model:', 'loaded and reusable' if status['model_loaded'] else 'loads on the first hint/check')
print('Results:', PERSISTENT / 'runs/ui' if PERSISTENT else WORK / 'outputs')
print('Server log:', UI_SERVER.log_path)
output.serve_kernel_port_as_iframe(int(UI_PORT), height=1100, cache_in_notebook=False)


## 9. Check server status and logs

Use this cell if startup or inference appears stuck. It reports the server state and recent logs without loading another model.


In [ ]:
if globals().get('UI_SERVER') is not None:
    if UI_SERVER.running:
        print(UI_SERVER.status())
    else:
        print('UI server is stopped.')
    print(UI_SERVER.logs())
else:
    print('UI has not been started in this session.')


## 10. Stop the UI when finished

Set STOP_UI=True and run this cell to release the model's GPU memory. False is the default so Run all leaves the UI running. Restarting later loads the model again.


In [ ]:
STOP_UI = False  # @param {type:"boolean"}
if STOP_UI and globals().get('UI_SERVER') is not None:
    UI_SERVER.stop()
    print('UI stopped; its model memory has been released.')
else:
    print('No stop requested. Set STOP_UI=True when finished with the browser UI.')


## 11. Download UI results

Set DOWNLOAD_RESULTS=True to download the local outputs/ui_*.json files as a ZIP. This also recovers local results after a failed Drive copy. With Drive enabled, successful jobs are already saved in My Drive/IE4422/runs/ui. Downloading is disabled by default.


In [ ]:
DOWNLOAD_RESULTS = False  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    from google.colab import files
    results = sorted((WORK / 'outputs').glob('ui_*.json'))
    if not results:
        raise RuntimeError('No saved UI results yet. Complete a hint or answer check first.')
    export = WORK / 'tmp/ie4422_ui_results.zip'
    export.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
        for result_path in results:
            archive.write(result_path, result_path.name)
    files.download(str(export))
else:
    print('Download disabled. Drive saving, when enabled, happens automatically.')


The model stays loaded until the server stops. Disk caches and saved results can survive in Drive; GPU memory and in-memory job status cannot. **Show inspection report** reveals reference answers and solutions, so leave it closed during student attempts. Model outputs are experimental and may be incorrect.
